# Prédictions U-Net sur l’échantillon de test
Choisir un modèle et un jeu de données, puis exécuter les cellules dans l’ordre.
Les prédictions sont la moyenne des cinq folds entraînés dans `eval_unet.ipynb`
ou, pour CSA-Net sur « 3classes », dans `eval_unet_2.5D.ipynb`.
Aucun entraînement n’est lancé. MultiResUNet n’est pas disponible dans les sources actuelles.

In [ ]:
import os
import sys
import random
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from PIL import Image
from matplotlib import pyplot as plt
import segmentation_models_pytorch as smp
from IPython.display import display

# Exécution depuis le dossier du notebook ou la racine du dépôt.
UNET_DIR = Path.cwd() if (Path.cwd() / "models/CSANet.py").exists() else Path.cwd() / "Debut/UNet"
UNET_DIR = UNET_DIR.resolve()
if not (UNET_DIR / "models/CSANet.py").exists():
    raise FileNotFoundError("Exécuter depuis Debut/UNet ou la racine du dépôt.")
sys.path.insert(0, str(UNET_DIR.parent))
sys.path.insert(0, str(UNET_DIR))
from utils.dataset_unet import RoboflowUNetDataset, EggHVDataset
from utils.dataset_2_5D import SliceSequenceUNetDataset
from utils.models_config import (collate_fn, collate_fn_egg_hv, collate_fn_sequence,
                                split_dataset, dataset_to_dict, metrics_by_class)
from utils.unet_spatial import unpad_array
from utils.stats_fct import overlay_mask, mask_to_yolo_polygons, plot_segmentation_comparison
from utils.post_traitement import keep_largest_components, fill_holes, split_eggs
from models.ResAttentionUNetPlusPlus import ResAttentionUNetPlusPlus
from models.CSANet import CSANet
from models.EggSegmentationHVUNet import EggSegmentationHVUNet, postprocess_egg_instances

### Configuration et reproductibilité

In [ ]:
MODEL_NAME = "EggSegmentationHVUNet"  # UNet, AttentionUNet++, EggSegmentationHVUNet, CSANet
DATASET_NAME = "oeufs"  # 2classes, 3classes, oeufs ; CSA-Net : 3classes uniquement
IGNORE = False  # Même convention que les notebooks d’évaluation
SEED = 42
IMAGE_SIZE = (512, 512)
THRESHOLD = 0.5
K_FOLDS = 5
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

SUPPORTED_DATASETS = {
    "UNet": {"2classes", "3classes", "oeufs"},
    "AttentionUNet++": {"2classes", "3classes", "oeufs"},
    "EggSegmentationHVUNet": {"oeufs"},
    "CSANet": {"3classes"},
}
if MODEL_NAME not in SUPPORTED_DATASETS or DATASET_NAME not in SUPPORTED_DATASETS[MODEL_NAME]:
    raise ValueError(f"Configuration non prise en charge : {MODEL_NAME} / {DATASET_NAME}. "
                     f"Configurations : {SUPPORTED_DATASETS}")
IS_SEQUENCE = MODEL_NAME == "CSANet"
IS_HV = MODEL_NAME == "EggSegmentationHVUNet"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Appareil : {device}, IGNORE={IGNORE}")

### Données et découpage commun par poisson

In [ ]:
DATASET_CONFIG = {
    "2classes": {"images": "images/cavite", "annotations": "labels/COCO/cavite/annotations_2_classes.json"},
    "3classes": {"images": "images/cavite", "annotations": "labels/COCO/cavite/annotations.json"},
    "oeufs": {"images": "images/oeufs", "annotations": "labels/COCO/oeufs/annotations.oeufs.json"},
}
config = DATASET_CONFIG[DATASET_NAME]
images_dir = str(UNET_DIR.parent / "data" / config["images"])
annotations_json = str(UNET_DIR.parent / "data" / config["annotations"])
df = pd.read_excel(UNET_DIR.parent / "utils/correspondance_echo_final.xlsx",
                   dtype={"annee": str, "image_id": str, "new_name_file": str})
all_samples, categories, num_classes, class_ids, ignore_category_id = dataset_to_dict(
    df, annotations_json, images_dir)
gonade_id = class_ids.index(next((cid for cid, name in categories.items() if name == "Gonade"),
                                class_ids[0])) if ignore_category_id is not None else None
class_names = [categories[cid] for cid in class_ids]
cv_samples, test_samples, cv_categories, test_categories, groups_train, groups_test = split_dataset(all_samples)
assert set(groups_train).isdisjoint(set(groups_test))
print(f"Modèle : {MODEL_NAME}, données : {DATASET_NAME}, classes : {class_names}")

In [ ]:
def build_dataset(samples):
    if IS_HV:
        if not class_ids:
            raise ValueError("Aucune catégorie COCO disponible pour le modèle HV.")
        return EggHVDataset(samples=samples, image_size=IMAGE_SIZE, is_train=False,
                            egg_category_id=class_ids[0]), collate_fn_egg_hv
    dataset_class = SliceSequenceUNetDataset if IS_SEQUENCE else RoboflowUNetDataset
    dataset = dataset_class(samples=samples, image_dir=images_dir, df=df, image_size=IMAGE_SIZE,
                            num_classes=num_classes, class_ids=class_ids, is_train=False,
                            ignore_category_id=ignore_category_id,
                            ignore_target_class_idx=gonade_id if IGNORE else None)
    return dataset, collate_fn_sequence if IS_SEQUENCE else collate_fn

test_dataset, test_collate = build_dataset(test_samples)
if not len(test_dataset):
    raise ValueError("Le dataset de test ne contient aucune image ou séquence exploitable.")
test_loader = DataLoader(test_dataset, batch_size=1 if IS_SEQUENCE else 4,
                         shuffle=False, collate_fn=test_collate)
nb_channels = test_dataset[0]["image"].shape[-3]
meta_by_name = {Path(s["image_info"]["file_name"]).name: s for s in test_samples}
echelle_cm_by_name = {}
for name in meta_by_name:
    rows = df.loc[df["image_id"] == name.split("_")[0], "echelle"]
    if rows.empty or not np.isfinite(float(rows.iloc[0])) or float(rows.iloc[0]) <= 0:
        raise ValueError(f"Échelle absente ou invalide : {name}")
    echelle_cm_by_name[name] = float(rows.iloc[0])
print(f"Forme du premier élément : {test_dataset[0]['image'].shape}")

### Chargement des cinq folds

In [ ]:
def build_model():
    # Les checkpoints contiennent tous les poids : aucun téléchargement ImageNet.
    if MODEL_NAME == "UNet":
        return smp.Unet(encoder_name="resnet34", encoder_weights=None,
                        in_channels=nb_channels, classes=num_classes, activation=None)
    if MODEL_NAME == "AttentionUNet++":
        return ResAttentionUNetPlusPlus(num_classes=num_classes, in_channels=nb_channels,
                                        encoder_weights=None)
    if IS_HV:
        return EggSegmentationHVUNet(in_channels=nb_channels, encoder_weights=None)
    return CSANet(num_classes=num_classes, in_channels=nb_channels, encoder_weights=None)

checkpoint_paths = [UNET_DIR / "saves" / f"unet_finetuned_fold_{fold}_{MODEL_NAME}_{DATASET_NAME}.pth"
                    for fold in range(1, K_FOLDS + 1)]
missing = [str(path) for path in checkpoint_paths if not path.is_file()]
if missing:
    raise FileNotFoundError("Checkpoints introuvables :\n" + "\n".join(missing))
models_ensemble = []
for path in checkpoint_paths:
    model = build_model()
    model.load_state_dict(torch.load(path, map_location="cpu", weights_only=True))
    models_ensemble.append(model.to(device).eval())
print(f"{len(models_ensemble)} folds chargés.")

### Inférence et post-traitement communs à toutes les visualisations

In [ ]:
def unpack_batch(batch):
    images = batch["image"].to(device)
    if IS_SEQUENCE:
        display_images = images.squeeze(0)
        targets = batch["mask"].to(device).squeeze(0)
        names = [Path(path).name for path in batch["image_path"][0]]
        sizes = batch["original_size"][0]
    else:
        display_images = images
        targets = batch["mask"].to(device)
        names = ([Path(path).name for path in batch["image_path"]]
                 if IS_HV else batch["file_names"])
        sizes = batch["original_size"]
    if IS_HV:
        targets = targets[:, :1]  # Les deux autres canaux sont les cibles horizontal/vertical.
    return images, display_images, targets, names, sizes

@torch.inference_mode()
def predict_batch(batch):
    images, display_images, targets, names, sizes = unpack_batch(batch)
    probs = torch.zeros_like(targets)
    hv_sum = None
    for model in models_ensemble:
        outputs = model(images)
        if IS_SEQUENCE:
            outputs = outputs.squeeze(0)
        if IS_HV:
            hv_sum = outputs.clone() if hv_sum is None else hv_sum + outputs
            outputs = outputs[:, :1]
        probs += torch.sigmoid(outputs)
    probs /= len(models_ensemble)
    valid = targets != -100
    true = (targets > THRESHOLD) & valid
    pred = (probs > THRESHOLD) & valid
    instances = None
    if DATASET_NAME == "oeufs":
        if IS_HV:
            instances = torch.stack([torch.from_numpy(postprocess_egg_instances(
                output, mask_threshold=THRESHOLD)[1]) for output in hv_sum / len(models_ensemble)]).to(device)
        else:
            instances = split_eggs(pred, min_distance=8, min_area=100,
                                   ouverture_size=3, ouverture_iterations=1)
        pred = (instances > 0).unsqueeze(1) & valid
    else:
        pred = fill_holes(keep_largest_components(pred)) & valid
    return display_images, true, pred, probs, instances, names, sizes

def batch_metrics(batch, true, pred, instances, names, sizes):
    scales = torch.tensor([echelle_cm_by_name[name] for name in names],
                          device=device, dtype=torch.float32).view(-1, 1)
    return metrics_by_class(
        true, pred, scales, DATASET_NAME,
        egg_instance_labels=instances if DATASET_NAME == "oeufs" else None,
        egg_annotation_areas_px=batch["egg_annotation_areas_px"] if DATASET_NAME == "oeufs" else None,
        image_height_px=[size[1] for size in sizes] if DATASET_NAME == "oeufs" else None,
    )

### Prédictions complètes, métriques et exports
Chaque ligne de résultats correspond à une image et une classe. Les noms et catégories suivent l’ordre réel des coupes prédites.

In [ ]:
results_dir = UNET_DIR / "results/test" / MODEL_NAME / DATASET_NAME
masks_dir = UNET_DIR / "masks/test" / MODEL_NAME / DATASET_NAME
results_dir.mkdir(parents=True, exist_ok=True)
masks_dir.mkdir(parents=True, exist_ok=True)
rows = []
seen_names = []
for batch in test_loader:
    images, true, pred, probs, instances, names, sizes = predict_batch(batch)
    intersection, union, iou, dice, precision, recall, diff_surface = batch_metrics(
        batch, true, pred, instances, names, sizes)
    true_surface = true.float().sum((2, 3))
    pred_surface = pred.float().sum((2, 3))
    ratio_surface = (true_surface - pred_surface).abs() / (true_surface + 1e-6)
    for i, name in enumerate(names):
        seen_names.append(name)
        sample = meta_by_name[name]
        for c, label in enumerate(class_names):
            rows.append({"name": name, "category": sample["category"], "classe": label,
                         "iou": iou[i, c].item(), "dice": dice[i, c].item(),
                         "precision": precision[i, c].item(), "recall": recall[i, c].item(),
                         "diff_surface": diff_surface[i, c].item(),
                         "ratio_surface": ratio_surface[i, c].item(),
                         "intersection": intersection[i, c].item(), "union": union[i, c].item(),
                         "true_surface_px": true_surface[i, c].item(),
                         "pred_surface_px": pred_surface[i, c].item()})
        stem = Path(name).stem
        # Dépadding avant affichage et export dans le référentiel cropé (510 × 380).
        image_crop = unpad_array(images[i]).cpu()
        true_crop = unpad_array(true[i]).cpu()
        pred_crop = unpad_array(pred[i]).cpu()
        combined = np.hstack((overlay_mask(image_crop, true_crop, alpha=0.4),
                              overlay_mask(image_crop, pred_crop, alpha=0.4)))
        Image.fromarray(combined).save(results_dir / f"{stem}.png")
        polygons = []
        for c in range(num_classes):
            polygons.extend(mask_to_yolo_polygons(unpad_array(pred[i, c]), c, target_size=(510, 380)))
        (masks_dir / f"pred_{stem}.txt").write_text("\n".join(polygons) + ("\n" if polygons else ""))

results_by_image = pd.DataFrame(rows)
if results_by_image.empty:
    raise ValueError("Aucune image testée.")
assert len(seen_names) == len(set(seen_names)), "Une image a été prédite plusieurs fois."
excluded_names = sorted(set(meta_by_name) - set(seen_names))
print(f"{len(seen_names)} images prédites ; {len(excluded_names)} images exclues par le dataset.")
if excluded_names:
    print("Images exclues (positions manquantes pour les séquences) :", excluded_names)
display(results_by_image)

### Résultats globaux et sauvegarde
Les différences de surface restent indéfinies (`NaN`) lorsqu’une surface réelle ou prédite est absente, comme dans les notebooks d’évaluation. Les moyennes ignorent ces valeurs.


In [ ]:
summary_rows = []
for label, frame in results_by_image.groupby("classe", sort=False):
    tp = frame["intersection"].sum()
    union = frame["union"].sum()
    predicted = frame["pred_surface_px"].sum()
    target = frame["true_surface_px"].sum()
    summary_rows.append({"classe": label, "mean_iou": frame["iou"].mean(),
                         "dataset_iou": tp / (union + 1e-6),
                         "dice": 2 * tp / (predicted + target + 1e-6),
                         "precision": tp / (predicted + 1e-6), "recall": tp / (target + 1e-6),
                         "mean_diff_surface": frame["diff_surface"].mean(),
                         "mean_ratio_surface": frame["ratio_surface"].mean()})
summary = pd.DataFrame(summary_rows).set_index("classe")
surface_unit = "mm²" if DATASET_NAME == "oeufs" else "cm²"
print(f"Différences de surface en {surface_unit}")
display(summary)
results_path = results_dir / f"results_test_{MODEL_NAME}_{DATASET_NAME}.pkl"
with results_path.open("wb") as f:
    pickle.dump({"model": MODEL_NAME, "dataset": DATASET_NAME, "ignore": IGNORE,
                 "threshold": THRESHOLD, "folds": K_FOLDS, "surface_unit": surface_unit,
                 "class_names": class_names, "results_by_image": results_by_image,
                 "summary": summary, "excluded_names": excluded_names}, f)
print(f"Résultats sauvegardés : {results_path}")

### Visualisation d’une image du test
Pour CSA-Net, la séquence complète du poisson est prédite avant de sélectionner la coupe.

In [ ]:
VIS_FILENAME = None  # Nom exact d’une image du test ; prioritaire sur VIS_INDEX
VIS_INDEX = 0  # Index dans l’ordre réel des images du dataset de test
VIS_CLASS = 0  # Index de classe pour la carte de probabilités
PROB_THRESHOLD = 0.3

ordered_names = []
locations = {}
for dataset_index in range(len(test_dataset)):
    item = test_dataset[dataset_index]
    paths = item["image_path"] if IS_SEQUENCE else [item["image_path"]]
    for slice_index, path in enumerate(paths):
        name = Path(path).name
        ordered_names.append(name)
        locations[name] = (dataset_index, slice_index)
if VIS_FILENAME is not None and str(VIS_FILENAME).strip():
    selected_name = Path(str(VIS_FILENAME).strip()).name
    if selected_name not in locations:
        raise ValueError(f"Image absente du test exploitable : {selected_name}")
else:
    if not 0 <= VIS_INDEX < len(ordered_names):
        raise IndexError(f"VIS_INDEX doit être compris entre 0 et {len(ordered_names) - 1}.")
    selected_name = ordered_names[VIS_INDEX]
if not 0 <= VIS_CLASS < num_classes:
    raise IndexError(f"VIS_CLASS doit être compris entre 0 et {num_classes - 1}.")
dataset_index, slice_index = locations[selected_name]
vis_batch = test_collate([test_dataset[dataset_index]])
vis_images, vis_true, vis_pred, vis_probs, vis_instances, vis_names, vis_sizes = predict_batch(vis_batch)
vis_metrics = batch_metrics(vis_batch, vis_true, vis_pred, vis_instances, vis_names, vis_sizes)
_, _, iou, dice, precision, recall, diff_surface = vis_metrics
display(pd.DataFrame([{"classe": label, "iou": iou[slice_index, c].item(),
                       "dice": dice[slice_index, c].item(), "precision": precision[slice_index, c].item(),
                       "recall": recall[slice_index, c].item(), "diff_surface": diff_surface[slice_index, c].item()}
                      for c, label in enumerate(class_names)]).set_index("classe"))
print(selected_name)
plot_segmentation_comparison(vis_images[slice_index].cpu(), vis_true[slice_index].cpu(),
                             vis_pred[slice_index].cpu(), is_eggs=DATASET_NAME == "oeufs", alpha=0.3)

In [ ]:
probability = unpad_array(vis_probs[slice_index, VIS_CLASS]).cpu().numpy()
image = unpad_array(vis_images[slice_index]).cpu().numpy().transpose(1, 2, 0)
masked_probability = np.ma.masked_where(probability < PROB_THRESHOLD, probability)
fig, ax = plt.subplots(figsize=(8, 5))
ax.imshow(image)
im = ax.imshow(masked_probability, cmap="turbo", vmin=PROB_THRESHOLD, vmax=1, alpha=0.5)
fig.colorbar(im, ax=ax)
ax.set_title(f"{selected_name} — {class_names[VIS_CLASS]} (seuil {PROB_THRESHOLD})")
ax.axis("off")
plt.tight_layout()
plt.show()